## Import necessary packages

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import hdf5plugin
import numpy as np
import anndata as ad
from scipy.sparse import csr_matrix
from CellPLM.utils import set_seed
from CellPLM.pipeline.cell_embedding import CellEmbeddingPipeline
import scanpy as sc
import matplotlib.pyplot as plt
# import rapids_singlecell as rsc  # For faster evaluation, we recommend the installation of rapids_singlecell.

## Specify important parameters before getting started

In [ ]:
PRETRAIN_VERSION = '20231027_85M'
DEVICE = 'cuda:0'

## Load Downstream Dataset

In [ ]:
set_seed(42)

In [ ]:
DATA_PATH = '/media/rokny/DATA2/Sally/data/srt/BC/breast_cancer.h5ad'

data = sc.read_h5ad(DATA_PATH) 

In [ ]:
# normalise gene expression
sc.pp.normalize_total(data, target_sum=1e4)

# log1p transformation
sc.pp.log1p(data)

In [ ]:
import mygene
import pandas as pd

# Initialize MyGeneInfo
mg = mygene.MyGeneInfo()

# Get your gene symbols from AnnData
gene_symbols = data.var_names.tolist()

# Query MyGene for Ensembl gene IDs
out = mg.querymany(
    gene_symbols,
    scopes="symbol",      # input type
    fields="ensembl.gene", # output field
    species="human"        # or "mouse" if needed
)

# Convert results into DataFrame for easy mapping
df = pd.DataFrame(out)

# Some genes return multiple Ensembl IDs; keep the first one
df["ensembl_id"] = df["ensembl"].apply(
    lambda x: x[0]["gene"] if isinstance(x, list) else (x["gene"] if isinstance(x, dict) else None)
)

# Handle missing Ensembl IDs (replace None with original gene symbol or drop them)
df["ensembl_id"].fillna(df["query"], inplace=True)

# Build mapping dictionary: {symbol -> ensembl_id}
mapping = df.set_index("query")["ensembl_id"].to_dict()

# Now update the var_names with Ensembl IDs
data.var["gene_symbol"] = data.var_names
data.var_names = [mapping.get(g, g) for g in data.var_names]

# Check for duplicates after the gene name conversion
if not data.var_names.is_unique:
    keep = ~data.var_names.duplicated(keep='first')
    data = data[:, keep].copy()

## Set up the pipeline

In [ ]:
pipeline = CellEmbeddingPipeline(pretrain_prefix=PRETRAIN_VERSION, # Specify the pretrain checkpoint to load
                                 pretrain_directory='../ckpt')
pipeline.model

In [ ]:
embedding = pipeline.predict(data, # An AnnData object
                device=DEVICE) # Specify a gpu or cpu for model inference

data.obsm['emb'] = embedding.cpu().numpy()
emb = data.obsm['emb']

## Save embeddings

In [ ]:
output_path = './embeddings/BC_with_zeroshot_embeddings.h5ad'

data.write_h5ad(output_path)

print(f"Embeddings saved to {output_path} in obsm['emb']")

## Spatial Plot

In [ ]:
# Set a seed for reproducibility

import os
import torch
import random

def fix_seed(seed):
    # Fix for Python hash seed (to ensure reproducibility in Python operations)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # Fix for random seed (used by random module)
    random.seed(seed)
    
    # Fix for numpy random operations
    np.random.seed(seed)
    
    # Fix for PyTorch random seed (CPU and GPU)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    
    # Ensure deterministic behavior for CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Additional configuration for CUBLAS for reproducibility
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    
    # For scanpy or any random process in other libraries
    # We can use a random_state argument where applicable, like in scanpy PCA, DEG, etc.

# Set the seed
seed = 42
fix_seed(seed)

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

# Read the AnnData objects
data = sc.read_h5ad("./embeddings/BC_with_zeroshot_embeddings.h5ad")

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from sklearn.decomposition import PCA


def clustering(adata, n_clusters=7, key='emb', method='leiden', start=0.1, end=3.0, increment=0.01):
    """\
    Spatial clustering based the learned representation.

    Returns
    -------
    None.

    """
    
    pca = PCA(n_components=20, random_state=seed) 
    embedding = pca.fit_transform(adata.obsm[key].copy())
    adata.obsm['emb_pca'] = embedding
    
    if method == 'leiden':
       res = search_res(adata, n_clusters, use_rep='emb_pca', method=method, start=start, end=end, increment=increment)
       sc.tl.leiden(adata, random_state=seed, resolution=res)
       adata.obs['domain'] = adata.obs['leiden']
    elif method == 'louvain':
       res = search_res(adata, n_clusters, use_rep='emb_pca', method=method, start=start, end=end, increment=increment)
       sc.tl.louvain(adata, random_state=seed, resolution=res)
       adata.obs['domain'] = adata.obs['louvain'] 
       
    

def extract_top_value(map_matrix, retain_percent = 0.1): 
    '''\
    Filter out cells with low mapping probability

    Parameters
    ----------
    map_matrix : array
        Mapped matrix with m spots and n cells.
    retain_percent : float, optional
        The percentage of cells to retain. The default is 0.1.

    Returns
    -------
    output : array
        Filtered mapped matrix.

    '''

    #retain top 1% values for each spot
    top_k  = retain_percent * map_matrix.shape[1]
    output = map_matrix * (np.argsort(np.argsort(map_matrix)) >= map_matrix.shape[1] - top_k)
    
    return output 
    
def search_res(adata, n_clusters, method='leiden', use_rep='emb', start=0.1, end=3.0, increment=0.01):
    '''\
    Searching corresponding resolution according to given cluster number
    
    Parameters
    ----------
    adata : anndata
        AnnData object of spatial data.
    n_clusters : int
        Targetting number of clusters.
    method : string
        Tool for clustering. Supported tools include 'leiden' and 'louvain'. The default is 'leiden'.    
    use_rep : string
        The indicated representation for clustering.
    start : float
        The start value for searching.
    end : float 
        The end value for searching.
    increment : float
        The step size to increase.
        
    Returns
    -------
    res : float
        Resolution.
        
    '''
    print('Searching resolution...')
    label = 0
    sc.pp.neighbors(adata, n_neighbors=50, use_rep=use_rep, random_state=seed)
    for res in sorted(list(np.arange(start, end, increment)), reverse=True):
        if method == 'leiden':
           sc.tl.leiden(adata, random_state=seed, resolution=res)
           count_unique = len(pd.DataFrame(adata.obs['leiden']).leiden.unique())
           print('resolution={}, cluster number={}'.format(res, count_unique))
        elif method == 'louvain':
           sc.tl.louvain(adata, random_state=seed, resolution=res)
           count_unique = len(pd.DataFrame(adata.obs['louvain']).louvain.unique()) 
           print('resolution={}, cluster number={}'.format(res, count_unique))
        if count_unique == n_clusters:
            label = 1
            break

    assert label==1, "Resolution is not found. Please try bigger range or smaller step!." 
       
    return res    


In [ ]:
# Run Leiden clustering 

n_clusters = 20
tool='leiden'

clustering(data, n_clusters, key='emb', method=tool, start=0.1, end=2.28, increment=0.01)

In [ ]:
labels = data.obs['leiden'].astype(int)

In [ ]:
p=['#1f77b4',
 '#aec7e8',
 '#ff7f0e',
 '#ffbb78',
 '#2ca02c',
 '#98df8a',
 '#d62728',
 '#ff9896',
 '#9467bd',
 '#c5b0d5',
 '#8c564b',
 '#c49c94',
 '#e377c2',
 '#f7b6d2']

In [ ]:
import matplotlib.pyplot as plt

Model_name='cellplm'
step='zero_shot' # or 'zero_shot'
dataset='BC' # or 'BC' or 'mouse_slideseq'

plt.rcParams["figure.figsize"] = (3,3)
sc.pl.embedding(data, basis="spatial", color="domain", palette=p, show=False, title='')
plt.gca().invert_yaxis()  # This will invert the y-axis
plt.axis('off')
plt.savefig(f"./figures/{dataset}_{Model_name}_leiden_{step}.png",dpi=600,bbox_inches="tight")
plt.savefig(f"./figures/{dataset}_{Model_name}_leiden_{step}.svg",bbox_inches="tight")
plt.show()

## ARI, NMI & Silhouette Scores

In [ ]:
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score, silhouette_score

ari_score = adjusted_rand_score(data.obs['leiden'].to_numpy(), data.obs['fine_annot_type'].to_numpy())
nmi_score = normalized_mutual_info_score(data.obs['leiden'].to_numpy(), data.obs['fine_annot_type'].to_numpy())
sil_score = silhouette_score(data.obsm['emb'], data.obs['leiden'].astype(int))

print(f"'ari': {ari_score}, 'nmi': {nmi_score}, 'sil': {sil_score}")

## Save results to npz file

In [ ]:
import numpy as np

ARI, NMI, SIL = float(ari_score), float(nmi_score), float(sil_score)

# --- SAVE (one compact file per model) ---
np.savez_compressed(
    f"./benchmarking_results/{Model_name}_{step}_clusters_{dataset}.npz",
    labels=labels,      
    embeddings=data.obsm['emb'],    
    ARI=ARI, NMI=NMI, SIL=SIL,                  
    coords=data.obsm["spatial"]
)